# 02 — Data preparation and supervised fine-tuning (Task 3)

Accelerator: **GPU T4 x2** (SFT uses `cuda:0`). Prepares all fixed train/val subsets (seed 42), runs a
one-minute tiny-model pipeline check, then trains SFT with the guide configuration in `configs/sft.yaml`.
Run with **Save Version → Save & Run All (Commit)** so the ~1.5 h job survives the browser closing.

In [ ]:
# Clone (or update) the project repo into /kaggle/working. Notebook settings: Internet ON.
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")

In [ ]:
# Manual section 5.2: keep Kaggle's PyTorch; do not reinstall it.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
# Manual section 7: confirm two T4 GPUs and print package versions.
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
sh("PYTHONPATH=. python -m pytest -q tests/")

In [ ]:
# UltraChat (SFT), single-turn HH-RLHF and UltraFeedback (DPO). Writes data/processed/*.jsonl
# and ID manifests + filter statistics to data/manifests/ (commit those to the repo afterwards).
sh("python scripts/prepare_data.py --out_dir data/processed --which sft hh uf")
!wc -l data/processed/*.jsonl
!cat data/manifests/*_stats.json

In [ ]:
sh("python scripts/smoke_train.py --work_dir /tmp/pa2_smoke")

In [ ]:
# Optional short pilot: uncomment to check speed/memory before the full run.
# !python scripts/train_sft.py --config configs/sft.yaml --set max_steps=20 eval_every=10 output_dir=outputs/sft_pilot

In [ ]:
sh("python scripts/train_sft.py --config configs/sft.yaml")

In [ ]:
!cat outputs/sft/summary.json
!python scripts/plot_metrics.py --run sft=outputs/sft --out_dir outputs/plots
from IPython.display import Image, display
display(Image("outputs/plots/sft_loss.png"))

In [ ]:
# Manual section 5.5: archive the run for easy attaching/downloading, then use "Save Version".
!cd outputs && zip -q -r -0 /kaggle/working/sft_run_outputs.zip sft
!ls -lh /kaggle/working/*.zip
!du -sh outputs/sft